# Crop Irrigation Need Prediction

Predicts irrigation need (`Low` / `Medium` / `High`) for a field from soil, weather, and crop features using a LightGBM classifier. See [README.md](README.md) for the full write-up.

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

mapping = {"High":2, "Medium":1, "Low":0}
train_df['Irrigation_Need'] = train_df['Irrigation_Need'].map(mapping)

hot_encode = ['Region', 'Water_Source', 'Irrigation_Type', 'Season', "Crop_Growth_Stage", "Crop_Type", "Soil_Type"]
train_df = pd.get_dummies(train_df, columns=hot_encode, dtype=float)
test_df = pd.get_dummies(test_df, columns=hot_encode, dtype=float)

mulch_Map = {'Yes':1, 'No':0}
train_df["Mulching_Used"] = train_df["Mulching_Used"].map(mulch_Map)
test_df["Mulching_Used"] = test_df["Mulching_Used"].map(mulch_Map)


In [2]:
X = test_df

X_col = train_df.columns.drop('Irrigation_Need')
X_train = train_df[X_col]
y_train = train_df.Irrigation_Need

split = int(len(X_train)* 0.99)
X_val = X_train[split:]
y_val = y_train[split:]

X_train = X_train[:split]
y_train = y_train[:split]

In [3]:
X_train

,id,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Crop_Type_Cotton,Crop_Type_Maize,Crop_Type_Potato,Crop_Type_Rice,Crop_Type_Sugarcane,Crop_Type_Wheat,Soil_Type_Clay,Soil_Type_Loamy,Soil_Type_Sandy,Soil_Type_Silt
0,0,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,1,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,2,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,3,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,4,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
623695,623695,6.32,31.56,0.41,3.47,28.00,71.32,732.55,5.32,15.22,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
623696,623696,6.33,35.86,1.32,2.02,39.77,52.02,1415.61,7.48,15.96,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
623697,623697,6.42,14.47,0.34,0.49,20.57,27.96,1471.02,5.70,9.24,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
623698,623698,7.41,27.36,1.50,0.50,30.74,42.07,2305.59,7.30,16.05,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [4]:
y_val

623700    1
623701    1
623702    0
623703    0
623704    1
         ..
629995    1
629996    1
629997    2
629998    0
629999    0
Name: Irrigation_Need, Length: 6300, dtype: int64

In [5]:
print(train_df[split:].groupby("Irrigation_Need")["Irrigation_Need"].value_counts())

Irrigation_Need
0    3755
1    2355
2     190
Name: count, dtype: int64


In [6]:
print(X_train.shape)
print(test_df.shape)

(623700, 43)
(270000, 43)


In [7]:
import lightgbm as lgb


model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate = 0.05,
    max_depth=8,              # how deep each tree can go
    num_leaves=63,            # complexity of each tree
    subsample=0.8,            # use 80% of rows per tree (prevents overfitting)
    colsample_bytree=0.8,     # use 80% of columns per tree (prevents overfitting)
    class_weight={0:1, 1:1.5, 2:17},  # same imbalance fix as before
    n_jobs=-1,                # use all CPU cores
    random_state=42
    )
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

predictions = model.predict(X)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004045 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2985
[LightGBM] [Info] Number of data points in the train set: 623700, number of used features: 43
[LightGBM] [Info] Start training from score -1.077152
[LightGBM] [Info] Start training from score -1.107889
[LightGBM] [Info] Start training from score -1.111149
Training until validation scores don't improve for 50 rounds


[100]	valid_0's multi_logloss: 0.0957401


[200]	valid_0's multi_logloss: 0.0819128


[300]	valid_0's multi_logloss: 0.0769391


[400]	valid_0's multi_logloss: 0.0740156


[500]	valid_0's multi_logloss: 0.0719087
Did not meet early stopping. Best iteration is:
[500]	valid_0's multi_logloss: 0.0719087


In [8]:
from sklearn.metrics import classification_report, accuracy_score

val_predictions = model.predict(X_val)
print(f"Validation accuracy: {accuracy_score(y_val, val_predictions):.4f}")
print(classification_report(y_val, val_predictions, target_names=["Low", "Medium", "High"]))

Validation accuracy: 0.9798
              precision    recall  f1-score   support

         Low       0.98      0.99      0.99      3755
      Medium       0.98      0.97      0.97      2355
        High       0.90      0.92      0.91       190

    accuracy                           0.98      6300
   macro avg       0.96      0.96      0.96      6300
weighted avg       0.98      0.98      0.98      6300



In [ ]:
submissions = pd.DataFrame({'id': test_df.id, 'Irrigation_Need': predictions})
mapping = {2:"High", 1:"Medium", 0:"Low"}
submissions['Irrigation_Need'] = submissions['Irrigation_Need'].map(mapping)
submissions.to_csv('submission.csv', index=False)